In [46]:
from rdflib import Graph
import pandas as pd
from rdflib import URIRef,Literal
from rdflib.namespace import RDF,RDFS,Namespace,OWL,SKOS

This notebook loads fairops.ttl and papers.ttl and automatically populates the indiv.ttl with elements from Fairness_Metrics.xlsx (Notions, Metrics, Mitigation Techniques and Legal Requirements)

In [47]:
# Creating a copy of a given ttl file
gc = Graph()
gc.parse("./docs/fairops.ttl", format='turtle')
print(f"len(gc)={len(gc)}")
gp = Graph()
gp.parse("./docs/papers.ttl", format='turtle')
print(f"len(gp)={len(gp)}")
gi = Graph()
gi.parse("./infiles/empty_indiv.ttl", format='turtle')
print(f"len(gi)={len(gi)}")
CORE = Namespace(str(dict(gi.namespaces()).get("core")))
PAPERS = Namespace(str(dict(gi.namespaces()).get("papers")))
INDIV = Namespace(str(dict(gi.namespaces()).get("indiv")))
gi.bind("indiv",INDIV)
print(gi.serialize(format="turtle"))
#print(OWL.NamedIndividual)

len(gc)=1123
len(gp)=40697
len(gi)=3
@prefix core: <https://purl.org/fairops/core#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix papers: <https://purl.org/fairops/papers#> .

<https://purl.org/fairops/indiv> a owl:Ontology ;
    owl:imports core:,
        papers: .




In [48]:
#load the content of the METRICS excel sheet
df = pd.read_excel("../Fairness-Metrics.xlsx", sheet_name="METRICS",header=0, dtype=str)
df.fillna('', inplace=True)

def camelCaseMetric(s):
    temp = s.replace('_', ' ').replace('-', ' ').split('(')[0]
    temp = ' '.join([w.title() if w.islower() else w for w in temp.split()])
    temp=temp.replace(' ', '')
    #res = temp[0].lower() + temp[1:]
    return temp

df['Metric']=df['Metric'].apply(lambda x: camelCaseMetric(x))
#with pd.option_context('display.max_rows', None):
#    print(df['Metric'])
df.head(2)

,Metric,same as,Definition,Formula,Latex Description,Notion/Metric,Measures,Is measured by,Procedural/Outcome,Individual/Group,...,refers To (AI Task),applicable to LLMs,AI Type Of Use,Proposed by,Used by,Equivalent to,Conflicts with,Extends,Other relations,Notes
0,FairnessThroughUnawareness,,An algorithm is fair as long as any protected ...,Remove the sensitive features from the trainin...,An algorithm is fair as long as any protected ...,notion,NONE,NONE,procedural,not applicable,...,"Classification, TextGeneration",yes,"Conversation, Prediction, Recommendation",2016_GRGIC_HLACA,NONE,NONE,Any other fairness measure that considers the ...,NONE,NONE,"not actually a measure, might increase bias in..."
1,FairnessThroughAwareness,Individual fairness,An algorithm is fair if it gives similar predi...,"𝐷 (𝑀 (𝑥1), 𝑀 (𝑥2)) ≤ 𝑑 (𝑥1, 𝑥2), ∀𝑥1, 𝑥2 ∈ 𝐸 \...",An algorithm is fair if it gives similar predi...,notion,not applicable,"Total variation norm, Relative l-infinite metr...",outcome,individual,...,Classification,no,"Prediction, Recommendation",2012_DWORK,"2025_GAO, 2018_YONA, 2020_YUROCHKIN, 2019_LAHOTI",Individual fairness,NONE,NONE,NONE,


In [49]:
def checkExistence(gra,element,className):
    q = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    ASK {{
        <{element}> rdf:type ?type .
        ?type rdfs:subClassOf* <{className}> .
    }}
    """
    result = gra.query(q)
    return bool(result)


def add_common_elements_for_new_metric_notion(row):
    mURI=INDIV[camelCaseMetric(row["Metric"].strip())]
    gi.add((mURI, RDF.type, OWL.NamedIndividual))
    if row['same as'] != '':
        for el in row['same as'].split(','):
            gi.add((mURI, OWL.sameAs, INDIV[camelCaseMetric(el.strip())] )) 
    if row['Latex Description'] != '':
        gi.add((mURI, CORE.scientificArtifactDescription, Literal(row['Latex Description'])))
    
    ## Application Domain (a.k.a. Economic Sector Or Use Area -->Skipped because we use a SWRL rule from the papers now.
    '''for ad in row['Application Domain'].split(","):
        if ad!="NONE":
            #check if the App Domain is in core. If not, print an alert msg
            if not any(gc.triples((ad,RDF.type,CORE.ApplicationDomain))):
                print(f"{ad} is not an existing ApplicationDomain")'''
    
    ## AI Type Of Use. "is suitable for" relation
    for aiT in row['AI Type Of Use'].split(","):
        aiT=aiT.strip()
        if not aiT in ["NONE", ""] :
            #check if the AI Task is in core:. If not, print an alert msg
            if not checkExistence(gc,CORE[aiT],CORE.AITypeOfUse):
                print(f'The AI Type Of Use. {aiT} does not exist in core ontology')
            else:
                gi.add((mURI, CORE.isSuitableFor, CORE[aiT]))

    ## refersTo (AI Task). Previously was "Is suitable for... one or more AI Capabilities"
    for aiT in row['refers To (AI Task)'].split(","):
        aiT=aiT.strip()
        if not aiT in ["NONE", ""] :
            #check if the AI Task is in core:. If not, print an alert msg
            if not checkExistence(gc,CORE[aiT],CORE.AITask):
                print(f'The AITask {aiT} does not exist in core ontology')
            else:
                gi.add((mURI, CORE.refersTo, CORE[aiT]))
    
    #is proposed by 
    for paperKey in row['Proposed by'].split(",") :
        paperKey=paperKey.strip()
        if not paperKey in ["NONE", ""] :
            if not checkExistence(gp,PAPERS[paperKey],CORE.ScientificPaper):
                print(f'The ScientificPaper {paperKey} does not exist in core ontology')
            else:
                gi.add((mURI, CORE.isProposedBy, PAPERS[paperKey]))
    #is used by 
    for paperKey in row['Used by'].split(",") :
        paperKey=paperKey.strip()
        if not paperKey in ["NONE", ""] :
            if not checkExistence(gp,PAPERS[paperKey],CORE.ScientificPaper):
                print(f'The ScientificPaper {paperKey} does not exist in core ontology')
            else:
                gi.add((mURI, CORE.isUsedBy, PAPERS[paperKey]))
    # is equivalent to
    for el in row['Equivalent to'].split(','):
        el=el.strip()
        if not el in ["NONE", ""] :
            gi.add((mURI, CORE.isEquivalentTo, INDIV[camelCaseMetric(el)]))
    #extends
    for el in row['Extends'].split(','):
        el=el.strip()
        if not el in ["NONE", ""] :
            gi.add((mURI, CORE.extends, INDIV[camelCaseMetric(el)]))
    #notes
    if row['Notes'].strip() != '':
        gi.add((mURI, RDFS.comment, Literal(row['Notes'])))

    return mURI

#ADDING NOTIONS
def add_new_notion(row):
    mURI=add_common_elements_for_new_metric_notion(row)
    '''mURI=INDIV[camelCaseMetric(row["Metric"].strip())]
    gi.add((mURI, RDF.type, OWL.NamedIndividual))
    if row['same as'] != '':
        for el in row['same as'].split(','):
            gi.add((mURI, OWL.sameAs, INDIV[camelCaseMetric(el.strip())] )) 
    if row['Latex Description'] != '':
        gi.add((mURI, CORE.scientificArtifactDescription, Literal(row['Latex Description'])))'''
    if row['Procedural/Outcome'] == 'procedural':
        gi.add((mURI, RDF.type, CORE.ProceduralFairnessNotion))
    elif row['Procedural/Outcome'] == 'outcome':
        if row['Causal/Observational'] == 'causal' :
            gi.add((mURI, RDF.type, CORE.CausalFairnessNotion))
        else:
            gi.add((mURI, RDF.type, CORE.ObservationalFairnessNotion))
    
        if 'individual' in row['Individual/Group'] :
            gi.add((mURI, RDF.type, CORE.IndividualFairnessNotion))
        else:
            if 'independence' in row['Independence/Separation/Sufficiency'] :
                gi.add((mURI, RDF.type, CORE.IndependenceFairnessNotion))
            if 'separation' in row['Independence/Separation/Sufficiency'] :
                gi.add((mURI, RDF.type, CORE.SeparationFairnessNotion))
            if 'sufficiency' in row['Independence/Separation/Sufficiency'] :
                gi.add((mURI, RDF.type, CORE.SufficiencyFairnessNotion))
    

#ADDING METRICS   
def add_new_metric(row):
    mURI=add_common_elements_for_new_metric_notion(row)
    if 'calibration' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.CalibrationBasedFairnessMetric))
    if 'confusion matrix' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.ConfusionMatrixBasedFairnessMetric))
    if 'distance-based' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.DistanceBasedFairnessMetric))
    if 'item' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.ItemFocusedMetric))
    if 'user' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.UserFocusedMetric))
    if 'single' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.SingleGranularityMetric))
    if 'amortized' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.AmortizedGranularityMetric))
    if 'treatment' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.TreatmentFocusedMetric))
    if 'impact' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.ImpactFocusedMetric))
    if 'parity' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.ParityBasedFairnessMetric))
    if 'score' in row['Metric class'] :
        gi.add((mURI, RDF.type, CORE.ScoreBasedFairnessMetric))
    
    if 'Classifier' in row['LLM subcategory'] :
        gi.add((mURI, RDF.type, CORE.ClassifierFairnessMetric))
    if 'Distribution' in row['LLM subcategory'] :
        gi.add((mURI, RDF.type, CORE.DistributionFairnessMetric))
    if 'Lexicon' in row['LLM subcategory'] :
        gi.add((mURI, RDF.type, CORE.LexiconFairnessMetric))
    if 'Masked token' in row['LLM subcategory'] :
        gi.add((mURI, RDF.type, CORE.MaskedTokenFairnessMetric))
    if 'Pseudo-log likelihood' in row['LLM subcategory'] :
        gi.add((mURI, RDF.type, CORE.PseudoLogLikelihoodFairnessMetric))
    if 'Sentence embedding' in row['LLM subcategory'] :
        gi.add((mURI, RDF.type, CORE.SentenceEmbeddingFairnessMetric))
    if 'Word embedding' in row['LLM subcategory'] :
        gi.add((mURI, RDF.type, CORE.WordEmbeddingFairnessMetric))

    
    
    


#for index, row in df.iloc[0:20].iterrows():
for index, row in df.iterrows():
    if row["Notion/Metric"] == "notion": 
        add_new_notion(row)
    else:
        add_new_metric(row)

print(f"len(gi)={len(gi)}")


The AI Type Of Use. 2012_KAMISHIMA does not exist in core ontology
len(gi)=1020


In [50]:

#now that all new metrics and notions have been added, we can link them to each other
for index, row in df.iterrows():
    mURI=INDIV[camelCaseMetric(row["Metric"].strip())]
    if row["Notion/Metric"] == "metric":   #NOVEL METRIC
        for el in row['Measures'].split(','):
            if el != 'NONE' and el != '' and el != 'not applicable':
                gi.add((mURI, CORE.measures, INDIV[camelCaseMetric(el.strip())]))
    if row["Notion/Metric"] == "notion":   #NOVEL NOTION
        for el in row['Is measured by'].split(','):
            if el != 'NONE' and el != '' and el != 'not applicable':
                gi.add((mURI, CORE.isMeasuredBy, INDIV[camelCaseMetric(el.strip())]))

print(f"len(gi)={len(gi)}")





len(gi)=1094


In [51]:
# LOADING THE MITIGATION TECHNIQUES:
df = pd.read_excel("../Fairness-Metrics.xlsx", sheet_name="(Ont) Mitigation",header=0, dtype=str)
df.fillna('', inplace=True)

for index, row in df.iterrows():
    mtech_name=row['Mitigation Technique']
    mtechURI=INDIV[camelCaseMetric(mtech_name)]
        
    if (mtechURI, None, None) in gi:   #MIT. TECH. ALREADY IN THE ONTOLOGY
        print("This graph alreay contains triples about "+str(mtechURI))
        pass
    else:
        # Add the new mitigation technique to the graph
        gi.add((mtechURI, RDF.type, OWL.NamedIndividual))
        for category in row["A-J Category"].split(','):
            #gi.add((mtechURI, RDF.type, URIRef(prefix+category.strip())))
            gi.add((mtechURI, RDF.type, CORE[category.strip()]))
        if row['Description'] != '':
            gi.add((mtechURI, CORE.scientificArtifactDescription, Literal(row['Description'])))
        #add_new_scientific_paper(prefix=prefix,row=row,paper_col="Source")
        for paper in row['Source'].split(','):
            paperKey=paper.strip()
            if not paperKey in ["NONE", ""] :
                if not checkExistence(gp,PAPERS[paperKey],CORE.ScientificPaper):
                    print(f'The ScientificPaper {paperKey} does not exist in core ontology')
                else:
                    gi.add((mURI, CORE.isProposedBy, PAPERS[paperKey]))




In [52]:
# LOADING THE LEGAL REQUIREMENTS
df = pd.read_excel("../Fairness-Metrics.xlsx", sheet_name="Regulatory req.",header=0, dtype=str)
df.fillna('', inplace=True)

for index, row in df.iterrows():
    req_names=row['Requirement']
    for req_name in req_names.split(','):
        reqURI=INDIV[camelCaseMetric(req_name.replace('(','.').replace(')','').strip())]
        print(reqURI)
    if (reqURI, None, None) in gi:   #LEGAL REQ. ALREADY IN THE ONTOLOGY
        print("This graph alreay contains triples about "+str(reqURI))
        pass
    else:
        # Add the new req. to the graph
        gi.add((reqURI, RDF.type, OWL.NamedIndividual))
        for el in row['Type'].split(','):
            if el.strip() == "Management":
                gi.add((reqURI,RDF.type, CORE.ManagementLegalRequirement))
            elif el.strip() == "Technical":
                gi.add((reqURI,RDF.type, CORE.TechnicalLegalRequirement))

        #is part of
        gi.add((reqURI, URIRef('http://semanticscience.org/resource/SIO_000068'), CORE[row['Source'].replace(' ','_')]))
        #content and notes
        gi.add((reqURI, RDFS.comment, Literal(row['Content']+" Notes: "+row['Notes'])))
        #is operationalized by 
        for el in row['Notion'].split(','):
            elRef=INDIV[el.strip()] #URIRef(prefix+el.strip())
            if (elRef, None, None) in gi: # Notion already in the graph
                gi.add((reqURI, CORE.isOperationalizedBy, elRef ))
            else:
                print("Notion "+elRef+' not found')
        #itsComplianceIsSupportedWith
        for el in row['Mitigation tech.'].split(','):
            el=camelCaseMetric(el.strip())
            if not el in ['', 'N.A.']:
                elRef= INDIV[el]
                if (elRef, None, None) in gi: # Mitigation tech. already in the graph
                    gi.add((reqURI, CORE.itsComplianceIsSupportedWith, elRef ))
                else:
                    print("Mitigation tech. "+elRef+' not found')
        
gi.serialize(destination="./docs/indiv.ttl")


https://purl.org/fairops/indiv#Art.2
https://purl.org/fairops/indiv#Art.3
https://purl.org/fairops/indiv#Art.4
https://purl.org/fairops/indiv#Art.5
https://purl.org/fairops/indiv#Art.6
https://purl.org/fairops/indiv#Art.6.2
https://purl.org/fairops/indiv#Art.9
https://purl.org/fairops/indiv#Art.10
https://purl.org/fairops/indiv#Art.10.3
https://purl.org/fairops/indiv#Art.10.4
https://purl.org/fairops/indiv#Art.10.5
https://purl.org/fairops/indiv#Art.11
https://purl.org/fairops/indiv#Art.12
https://purl.org/fairops/indiv#Art.13
https://purl.org/fairops/indiv#Art.14
https://purl.org/fairops/indiv#Art.15
https://purl.org/fairops/indiv#Art.16
https://purl.org/fairops/indiv#Art.17
https://purl.org/fairops/indiv#Art.20
https://purl.org/fairops/indiv#Art.21
https://purl.org/fairops/indiv#Art.23
https://purl.org/fairops/indiv#Art.24
https://purl.org/fairops/indiv#Art.25
https://purl.org/fairops/indiv#Art.26
https://purl.org/fairops/indiv#Art.27
https://purl.org/fairops/indiv#Art.29
https://pur

<Graph identifier=N1efe98c120f246f4ae6196fa75ea4e06 (<class 'rdflib.graph.Graph'>)>